In [11]:
# ==============================================================================
# 0. CONFIGURATION
# ==============================================================================

# ── Paths ──────────────────────────────────────────────────────────────────────
CSV_PATH   = "df_final_binaire_imputed.csv"
OUTPUT_DIR = "Results/Regular_clustering/Full_dataset"

# ── Description features ───────────────────────────────────────────────────
MEASURED_STATUS_PAIRS = {
    "is_bp_measured":                     "bp_status",
    "is_ht_measured":                     "ht_status",
    "is_temp_measured":                   "temp_status",
    "is_sat_measured":                     "sat_status",
    "is_rr_measured":                     "rr_status",
    "is_o2_measured":                     "o2_flow_status",
    "is_gcs_measured":                    "gcs_status",
    "is_cap_blood_sugar_mmol_L_measured": "cap_blood_sugar_status",
    "is_pupil_right_measured":            "anisocoria_status",
    "is_urine_dipstick_clean_measured":   "urine_dipstick_clean_status",
    "is_pain_measured":                   "pain_status",
    "is_breathalyzer_measured":           "breathalyzer_status",
    "is_hemocue_measured":                "hemocue_status",
}

ADMISSION_QUANTI = [
    "age",
    "duration_triage_ioa_min",
]

ADMISSION_CATEG = [
    "age_group",
    "sex",
    "transport_grouped",
]

CHIEF_COMPLAINT_COL = "chief_complaint"
TOP_N_COMPLAINTS    = 10


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

import os
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

In [12]:
# ==============================================================================
# 1. CLUSTERS DESCRIPTION — EXTERNAL FEATURES (ADMISSION, TRIAGE, CHIEF COMPLAINT, IS_MEASURED, STATUS)
# ==============================================================================



# ==============================================================================
# 1. MAIN FUNCTION
# ==============================================================================

def describe_clusters(
    df:           pd.DataFrame,
    labels:       np.ndarray,
    idx:          pd.Index,
    run_label:    str,
    out_dir:      str,
    df_sub:       pd.DataFrame = None,
):
    """
    Generate a complete description of clusters :
    1. Clustering variables (heatmap already generated, here stats tables)
    2. Admission and triage variables

    Parameters
    ----------
    df        : Full DataFrame (all variables)
    labels    : HDBSCAN label array
    idx       : Index of patients used in clustering
    run_label : Run identifier
    out_dir   : Output directory
    df_sub    : DataFrame of clustering variables (optional)
    """
    os.makedirs(out_dir, exist_ok=True)

    # Rebuilding a df with labels and external variables for description
    df_desc           = df.loc[idx].copy()
    df_desc["cluster"] = labels
    df_desc           = df_desc[df_desc["cluster"] != -1]  # exclure le bruit

    n_total    = len(df_desc)
    clusters   = sorted(df_desc["cluster"].unique())
    n_clusters = len(clusters)

    log.info(f"[{run_label}] Description of {n_clusters} clusters ({n_total} patients)")

    # ── 1. Clusters size ─────────────────────────────────────────────────
    _plot_cluster_sizes(df_desc, clusters, run_label, out_dir)

    # ── 2. Quantitative variables from triage ───────────────────────────────────
    quanti_available = [c for c in ADMISSION_QUANTI if c in df_desc.columns]
    if quanti_available:
        _plot_quanti_by_cluster(df_desc, quanti_available, clusters, run_label, out_dir)

    # ── 3. Categorical variables from triage ──────────────────────────────────
    categ_available = [c for c in ADMISSION_CATEG if c in df_desc.columns]
    if categ_available:
        _plot_categ_by_cluster(df_desc, categ_available, clusters, run_label, out_dir)

    # ── 4. Triage (ordinal) ───────────────────────────────────────────────────
    if "triage" in df_desc.columns:
        _plot_triage_by_cluster(df_desc, clusters, run_label, out_dir)

    # ── 5. Chief complaint — top 10 per cluster ───────────────────────────────
    if CHIEF_COMPLAINT_COL in df_desc.columns:
        _plot_chief_complaint(df_desc, clusters, run_label, out_dir)

    # ── 6. Variables is_measured ──────────────────────────────────────────────
    _plot_measured_vars(df_desc, clusters, run_label, out_dir)

    # ── 7. Export summary table CSV ───────────────────────────────────
    _export_summary_table(df_desc, clusters, quanti_available, categ_available, run_label, out_dir)

    log.info(f"[{run_label}] Description done → {out_dir}")


# ==============================================================================
# 2. SUB FUNCTIONS FOR CLUSTER DESCRIPTION
# ==============================================================================

def _plot_cluster_sizes(df_desc, clusters, run_label, out_dir):
    sizes = df_desc["cluster"].value_counts().sort_index()
    pcts  = (sizes / sizes.sum() * 100).round(1)

    fig, ax = plt.subplots(figsize=(max(6, len(clusters) * 1.2), 5))
    bars = ax.bar(
        [f"C{c}" for c in sizes.index],
        sizes.values,
        color=sns.color_palette("tab10", len(clusters))
    )
    for bar, pct in zip(bars, pcts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + sizes.max() * 0.01,
            f"{pct}%", ha="center", va="bottom", fontsize=10
        )
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Number of patients")
    ax.set_title(f"Clusters size — {run_label}")
    plt.tight_layout()
    path = os.path.join(out_dir, "cluster_sizes.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"Saved: {path}")


def _plot_quanti_by_cluster(df_desc, quanti_cols, clusters, run_label, out_dir):
    n     = len(quanti_cols)
    fig, axes = plt.subplots(1, n, figsize=(n * 5, 5))
    if n == 1:
        axes = [axes]

    palette = sns.color_palette("tab10", len(clusters))

    for ax, col in zip(axes, quanti_cols):
        data = [df_desc[df_desc["cluster"] == c][col].dropna().values for c in clusters]
        bp = ax.boxplot(data, patch_artist=True, labels=[f"C{c}" for c in clusters])
        for patch, color in zip(bp["boxes"], palette):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_title(col)
        ax.set_xlabel("Cluster")

    plt.suptitle(f"Quantitatives variables — {run_label}", y=1.02)
    plt.tight_layout()
    path = os.path.join(out_dir, "admission_quanti.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"Saved: {path}")


def _plot_categ_by_cluster(df_desc, categ_cols, clusters, run_label, out_dir):
    for col in categ_cols:
        if col not in df_desc.columns:
            continue

        # Proportion de chaque modalité par cluster
        ct = (
            df_desc.groupby(["cluster", col])
            .size()
            .reset_index(name="n")
        )
        ct["pct"] = ct.groupby("cluster")["n"].transform(lambda x: x / x.sum() * 100)

        modalities = df_desc[col].dropna().unique()
        n_mod      = len(modalities)

        fig, ax = plt.subplots(figsize=(max(8, len(clusters) * 1.5), 5))
        x       = np.arange(len(clusters))
        width   = 0.8 / n_mod
        palette = sns.color_palette("tab10", n_mod)

        for i, mod in enumerate(sorted(modalities)):
            vals = []
            for c in clusters:
                row = ct[(ct["cluster"] == c) & (ct[col] == mod)]
                vals.append(row["pct"].values[0] if len(row) > 0 else 0)
            ax.bar(x + i * width, vals, width, label=str(mod), color=palette[i], alpha=0.8)

        ax.set_xticks(x + width * (n_mod - 1) / 2)
        ax.set_xticklabels([f"C{c}" for c in clusters])
        ax.set_ylabel("% patients")
        ax.set_title(f"{col} par cluster — {run_label}")
        ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        path = os.path.join(out_dir, f"admission_{col}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")


def _plot_triage_by_cluster(df_desc, clusters, run_label, out_dir):
    triage_levels = sorted(df_desc["triage"].dropna().unique())
    ct = (
        df_desc.groupby(["cluster", "triage"])
        .size()
        .reset_index(name="n")
    )
    ct["pct"] = ct.groupby("cluster")["n"].transform(lambda x: x / x.sum() * 100)

    n_levels = len(triage_levels)
    x        = np.arange(len(clusters))
    width    = 0.8 / n_levels
    palette  = sns.color_palette("RdYlGn_r", n_levels)

    fig, ax = plt.subplots(figsize=(max(8, len(clusters) * 1.5), 5))
    for i, level in enumerate(triage_levels):
        vals = []
        for c in clusters:
            row = ct[(ct["cluster"] == c) & (ct["triage"] == level)]
            vals.append(row["pct"].values[0] if len(row) > 0 else 0)
        ax.bar(x + i * width, vals, width,
               label=f"Triage {int(level)}", color=palette[i], alpha=0.85)

    ax.set_xticks(x + width * (n_levels - 1) / 2)
    ax.set_xticklabels([f"C{c}" for c in clusters])
    ax.set_ylabel("% patients")
    ax.set_title(f"Triage per cluster — {run_label}")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    path = os.path.join(out_dir, "admission_triage.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"Saved: {path}")


def _plot_chief_complaint(df_desc, clusters, run_label, out_dir):
    # Top 10 global
    top10 = (
        df_desc[CHIEF_COMPLAINT_COL]
        .value_counts()
        .head(TOP_N_COMPLAINTS)
        .index.tolist()
    )

    ct = (
        df_desc[df_desc[CHIEF_COMPLAINT_COL].isin(top10)]
        .groupby(["cluster", CHIEF_COMPLAINT_COL])
        .size()
        .reset_index(name="n")
    )
    ct["pct"] = ct.groupby("cluster")["n"].transform(lambda x: x / x.sum() * 100)

    # Un graphique par cluster
    n_cols_fig = min(3, len(clusters))
    n_rows_fig = (len(clusters) + n_cols_fig - 1) // n_cols_fig
    fig, axes  = plt.subplots(n_rows_fig, n_cols_fig,
                               figsize=(n_cols_fig * 6, n_rows_fig * 5))
    axes = np.array(axes).flatten()

    palette = sns.color_palette("tab10", TOP_N_COMPLAINTS)

    for i, c in enumerate(clusters):
        ax   = axes[i]
        data = ct[ct["cluster"] == c].sort_values("pct", ascending=True)
        ax.barh(data[CHIEF_COMPLAINT_COL], data["pct"],
                color=palette[:len(data)], alpha=0.8)
        ax.set_title(f"Cluster {c}")
        ax.set_xlabel("% patients")

    # Cacher les axes vides
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(f"Top {TOP_N_COMPLAINTS} chief complaints — {run_label}", y=1.02)
    plt.tight_layout()
    path = os.path.join(out_dir, "chief_complaint_by_cluster.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"Saved: {path}")


def _plot_measured_vars(df_desc, clusters, run_label, out_dir):
    available_pairs = {
        k: v for k, v in MEASURED_STATUS_PAIRS.items()
        if k in df_desc.columns
    }
    if not available_pairs:
        return

    # ── Measure rate per cluster ─────────────────────────────────────────────
    measured_rates = pd.DataFrame(index=[f"C{c}" for c in clusters])
    for is_col in available_pairs:
        rates = []
        for c in clusters:
            sub  = df_desc[df_desc["cluster"] == c][is_col]
            rates.append(sub.mean() * 100 if len(sub) > 0 else 0)
        measured_rates[is_col.replace("is_", "").replace("_measured", "")] = rates

    fig, ax = plt.subplots(figsize=(max(10, len(available_pairs) * 1.2), 5))
    sns.heatmap(
        measured_rates.T, annot=True, fmt=".1f", cmap="YlOrRd",
        ax=ax, annot_kws={"size": 9}
    )
    ax.set_title(f"Taux de mesure (%) par cluster — {run_label}")
    ax.set_xlabel("Cluster")
    plt.tight_layout()
    path = os.path.join(out_dir, "measured_rates.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"Saved: {path}")

    # ── Status per variable (only measured patients) ─────────────────────
    for is_col, status_col in available_pairs.items():
        if status_col not in df_desc.columns:
            continue

        var_name = is_col.replace("is_", "").replace("_measured", "")

        # Filter only measured patients
        df_measured = df_desc[df_desc[is_col] == 1].copy()
        if len(df_measured) == 0:
            continue

        modalities = df_measured[status_col].dropna().unique()
        if len(modalities) == 0:
            continue

        ct = (
            df_measured.groupby(["cluster", status_col])
            .size()
            .reset_index(name="n")
        )
        ct["pct"] = ct.groupby("cluster")["n"].transform(
            lambda x: x / x.sum() * 100
        )

        n_mod   = len(modalities)
        x       = np.arange(len(clusters))
        width   = 0.8 / n_mod
        palette = sns.color_palette("tab10", n_mod)

        fig, ax = plt.subplots(figsize=(max(8, len(clusters) * 1.5), 5))
        for i, mod in enumerate(sorted(modalities)):
            vals = []
            for c in clusters:
                row = ct[(ct["cluster"] == c) & (ct[status_col] == mod)]
                vals.append(row["pct"].values[0] if len(row) > 0 else 0)
            ax.bar(x + i * width, vals, width,
                   label=str(mod), color=palette[i], alpha=0.8)

        ax.set_xticks(x + width * (n_mod - 1) / 2)
        ax.set_xticklabels([f"C{c}" for c in clusters])
        ax.set_ylabel("% Measured patients")
        ax.set_title(f"{var_name} status (Measured patients) — {run_label}")
        ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        path = os.path.join(out_dir, f"status_{var_name}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")


def _export_summary_table(df_desc, clusters, quanti_cols, categ_cols, run_label, out_dir):
    """summary table CSV — mean/median quanti + % categ per cluster."""
    rows = []

    for c in clusters:
        sub  = df_desc[df_desc["cluster"] == c]
        row  = {"cluster": c, "n": len(sub), "pct_total": round(len(sub) / len(df_desc) * 100, 1)}

        # Quanti
        for col in quanti_cols:
            if col in sub.columns:
                row[f"{col}_mean"]   = round(sub[col].mean(), 1)
                row[f"{col}_median"] = round(sub[col].median(), 1)
                row[f"{col}_std"]    = round(sub[col].std(), 1)

        # Categ — dominant modality + % of this modality
        for col in categ_cols:
            if col in sub.columns:
                mode = sub[col].mode()
                row[f"{col}_mode"] = mode.iloc[0] if len(mode) > 0 else None
                row[f"{col}_mode_pct"] = round(
                    (sub[col] == row[f"{col}_mode"]).mean() * 100, 1
                ) if row[f"{col}_mode"] else None

        # Triage — distribution
        if "triage" in sub.columns:
            for level in sorted(df_desc["triage"].dropna().unique()):
                row[f"triage_{int(level)}_pct"] = round(
                    (sub["triage"] == level).mean() * 100, 1
                )

        # is_measured — Rate
        for is_col in MEASURED_STATUS_PAIRS:
            if is_col in sub.columns:
                var_name = is_col.replace("is_", "").replace("_measured", "")
                row[f"{var_name}_measured_pct"] = round(sub[is_col].mean() * 100, 1)

        rows.append(row)

    summary = pd.DataFrame(rows)
    path    = os.path.join(out_dir, "cluster_summary.csv")
    summary.to_csv(path, index=False)
    log.info(f"Saved: {path}")
    return summary

In [13]:
# ==============================================================================
# 2. OUTLIERS DESCRIPTION — EXTERNAL FEATURES (ADMISSION, TRIAGE, CHIEF COMPLAINT, IS_MEASURED, STATUS)
# ==============================================================================


def describe_outliers_external(
    df_clust:  pd.DataFrame,
    labels:    np.ndarray,
    idx:       pd.Index,
    run_label: str,
    out_dir:   str,
):
    """
    Description of outliers using external variables
    (admission, triage, chief complaint, is_measured, status variables).
    To be called from the description notebook.
    """
    os.makedirs(out_dir, exist_ok=True)

    df_work            = df_clust.loc[idx].copy()
    df_work["cluster"] = labels

    df_noise     = df_work[df_work["cluster"] == -1]
    df_clustered = df_work[df_work["cluster"] != -1]

    n_noise = len(df_noise)
    n_total = len(df_work)

    log.info(
        f"[{run_label}] Outliers : {n_noise} / {n_total} "
        f"({100*n_noise/n_total:.1f}%)"
    )

    if n_noise == 0:
        log.info("No outliers found.")
        return


    # ── Triage ─────────────────────────────────────────────────────────────────────
    if "triage" in df_noise.columns:

        # ── Graphique 1 — composition des outliers par triage ─────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        triage_dist = df_noise["triage"].value_counts().sort_index()
        triage_pct  = (triage_dist / triage_dist.sum() * 100).round(1)
        palette     = sns.color_palette("RdYlGn_r", len(triage_dist))

        axes[0].bar(
            [f"Triage {int(t)}" for t in triage_dist.index],
            triage_pct.values, color=palette, alpha=0.85
        )
        axes[0].set_ylabel("% outliers")
        axes[0].set_title("Triage distribution — outliers")

        rates = []
        for level in sorted(df_work["triage"].dropna().unique()):
            n_lev       = (df_work["triage"] == level).sum()
            n_noise_lev = (df_noise["triage"] == level).sum()
            rates.append({
                "triage":      int(level),
                "pct_outlier": round(100 * n_noise_lev / n_lev, 1) if n_lev > 0 else 0
            })
        df_rates = pd.DataFrame(rates)
        axes[1].bar(
            [f"Triage {t}" for t in df_rates["triage"]],
            df_rates["pct_outlier"],
            color=sns.color_palette("RdYlGn_r", len(df_rates)), alpha=0.85
        )
        axes[1].set_ylabel("% classified as noise")
        axes[1].set_title("Outlier rate by triage level")

        plt.suptitle(f"Triage — outliers — {run_label}")
        plt.tight_layout()
        path = os.path.join(out_dir, "outliers_triage.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

        # ── Graphique 2 — distribution cluster+outlier par niveau de triage ───────
        triage_levels   = sorted(df_work["triage"].dropna().unique())
        unique_clusters = sorted([c for c in df_work["cluster"].unique() if c != -1])

        # Construire un df : pour chaque niveau de triage →
        # % dans chaque cluster + % outliers
        rows_triage = []
        for level in triage_levels:
            df_level = df_work[df_work["triage"] == level]
            n_level  = len(df_level)
            if n_level == 0:
                continue
            row = {"triage": int(level), "n_total": n_level}
            for c in unique_clusters:
                row[f"cluster_{c}"] = round(
                    (df_level["cluster"] == c).sum() / n_level * 100, 1
                )
            row["outliers"] = round(
                (df_level["cluster"] == -1).sum() / n_level * 100, 1
            )
            rows_triage.append(row)

        df_triage_dist = pd.DataFrame(rows_triage).set_index("triage")

        # Stacked bar — un bar par niveau de triage
        cluster_cols_plot = [f"cluster_{c}" for c in unique_clusters]
        all_cols_plot     = cluster_cols_plot + ["outliers"]

        # Palette — clusters en tab20, outliers en gris
        n_clusters      = len(unique_clusters)
        cluster_palette = sns.color_palette("tab20", n_clusters) \
                          if n_clusters <= 20 \
                          else sns.color_palette("hsv", n_clusters)
        color_list      = list(cluster_palette) + ["lightgrey"]

        fig, ax = plt.subplots(figsize=(max(8, len(triage_levels) * 1.5), 6))
        bottom  = np.zeros(len(df_triage_dist))

        for col, color in zip(all_cols_plot, color_list):
            values = df_triage_dist[col].values
            ax.bar(
                [f"Triage {t}" for t in df_triage_dist.index],
                values, bottom=bottom,
                label=col.replace("cluster_", "C").replace("outliers", "Outliers"),
                color=color, alpha=0.85, edgecolor="white",
            )
            bottom += values

        ax.set_ylabel("% patients")
        ax.set_xlabel("Triage level")
        ax.set_title(
            f"Cluster + outlier distribution by triage level — {run_label}"
        )
        ax.legend(
            title="Cluster", bbox_to_anchor=(1.05, 1),
            loc="upper left", fontsize=8,
        )
        ax.set_ylim(0, 105)

        # Annoter le % outliers sur chaque barre
        for i, (level, row) in enumerate(df_triage_dist.iterrows()):
            pct_out = row["outliers"]
            n_tot   = row["n_total"]
            ax.text(
                i, 102,
                f"n={int(n_tot)}\n{pct_out}% noise",
                ha="center", va="bottom", fontsize=8,
            )

        plt.tight_layout()
        path = os.path.join(out_dir, "triage_cluster_distribution.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

        # Export CSV
        df_triage_dist["n_total"] = [
            (df_work["triage"] == t).sum() for t in df_triage_dist.index
        ]
        df_triage_dist.to_csv(
            os.path.join(out_dir, "triage_cluster_distribution.csv")
        )
        log.info(f"Saved: triage_cluster_distribution.csv")

    # ── Quantitative admission variables ───────────────────────────────────────
    quanti_available = [c for c in ADMISSION_QUANTI if c in df_noise.columns]
    if quanti_available:
        fig, axes = plt.subplots(1, len(quanti_available),
                                  figsize=(len(quanti_available) * 5, 5))
        if len(quanti_available) == 1:
            axes = [axes]
        for ax, col in zip(axes, quanti_available):
            ax.boxplot(
                [df_noise[col].dropna().values,
                 df_clustered[col].dropna().values],
                patch_artist=True,
                labels=["Outliers", "Clustered"]
            )
            ax.set_title(col)
        plt.suptitle(f"Quantitative variables — outliers vs clustered — {run_label}")
        plt.tight_layout()
        path = os.path.join(out_dir, "outliers_quanti.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

    # ── Categorical admission variables ────────────────────────────────────────
    categ_available = [c for c in ADMISSION_CATEG if c in df_noise.columns]
    for col in categ_available:
        modalities  = sorted(df_work[col].dropna().unique())
        n_mod       = len(modalities)
        x           = np.arange(n_mod)
        width       = 0.35

        noise_pct   = df_noise[col].value_counts(normalize=True).mul(100).reindex(modalities, fill_value=0)
        cluster_pct = df_clustered[col].value_counts(normalize=True).mul(100).reindex(modalities, fill_value=0)

        fig, ax = plt.subplots(figsize=(max(8, n_mod * 1.2), 5))
        ax.bar(x - width/2, noise_pct.values,   width, label="Outliers",  color="tab:red",  alpha=0.8)
        ax.bar(x + width/2, cluster_pct.values, width, label="Clustered", color="tab:blue", alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(modalities, rotation=45, ha="right")
        ax.set_ylabel("% patients")
        ax.set_title(f"{col} — outliers vs clustered — {run_label}")
        ax.legend()
        plt.tight_layout()
        path = os.path.join(out_dir, f"outliers_{col}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

    # ── Chief complaint ────────────────────────────────────────────────────────
    if CHIEF_COMPLAINT_COL in df_noise.columns:
        top10 = df_noise[CHIEF_COMPLAINT_COL].value_counts().head(TOP_N_COMPLAINTS)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(top10.index[::-1], top10.values[::-1],
                color="tab:red", alpha=0.8)
        ax.set_xlabel("Number of patients")
        ax.set_title(f"Top {TOP_N_COMPLAINTS} chief complaints — outliers — {run_label}")
        plt.tight_layout()
        path = os.path.join(out_dir, "outliers_chief_complaint.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

    # ── Measurement rates ──────────────────────────────────────────────────────
    available_measured = {k: v for k, v in MEASURED_STATUS_PAIRS.items()
                          if k in df_noise.columns}
    if available_measured:
        rates_noise   = {
            k.replace("is_", "").replace("_measured", ""): df_noise[k].mean() * 100
            for k in available_measured
        }
        rates_cluster = {
            k.replace("is_", "").replace("_measured", ""): df_clustered[k].mean() * 100
            for k in available_measured
        }
        df_rates = pd.DataFrame({
            "outliers":  rates_noise,
            "clustered": rates_cluster,
        }).round(1)

        fig, ax = plt.subplots(figsize=(max(10, len(df_rates) * 1.2), 5))
        sns.heatmap(
            df_rates.T, annot=True, fmt=".1f", cmap="YlOrRd",
            ax=ax, annot_kws={"size": 9}
        )
        ax.set_title(f"Measurement rates (%) — outliers vs clustered — {run_label}")
        plt.tight_layout()
        path = os.path.join(out_dir, "outliers_measured_rates.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

    # ── Status variables ───────────────────────────────────────────────────────
    status_cols = [
        c for c in df_noise.columns
        if c.endswith("_status")
        and c in df_clustered.columns
    ]

    for col in status_cols:
        modalities = sorted(
            set(df_noise[col].dropna().unique()) |
            set(df_clustered[col].dropna().unique())
        )
        if len(modalities) == 0:
            continue

        n_mod  = len(modalities)
        x      = np.arange(n_mod)
        width  = 0.35

        noise_pct   = (
            df_noise[col].value_counts(normalize=True)
            .mul(100)
            .reindex(modalities, fill_value=0)
        )
        cluster_pct = (
            df_clustered[col].value_counts(normalize=True)
            .mul(100)
            .reindex(modalities, fill_value=0)
        )

        # Color map — not_measured = grey, invalid = orange
        color_map_status = {
            "not_measured": "lightgrey",
            "invalid":      "orange",
            "unknown":      "silver",
        }
        default_palette = sns.color_palette("tab10", n_mod)
        bar_colors = [
            color_map_status.get(m, default_palette[i])
            for i, m in enumerate(modalities)
        ]

        fig, axes = plt.subplots(1, 2, figsize=(max(12, n_mod * 1.5), 5),
                                  sharey=False)

        axes[0].bar(
            x, noise_pct.values,
            color=bar_colors, alpha=0.85, edgecolor="white"
        )
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(modalities, rotation=45, ha="right")
        axes[0].set_ylabel("% outliers")
        axes[0].set_title("Outliers")

        axes[1].bar(
            x, cluster_pct.values,
            color=bar_colors, alpha=0.85, edgecolor="white"
        )
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(modalities, rotation=45, ha="right")
        axes[1].set_ylabel("% clustered")
        axes[1].set_title("Clustered")

        plt.suptitle(
            f"{col} — outliers vs clustered — {run_label}",
            y=1.02
        )
        plt.tight_layout()
        path = os.path.join(out_dir, f"outliers_status_{col}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        log.info(f"Saved: {path}")

    # ── Export CSV ─────────────────────────────────────────────────────────────
    rows = []
    for group, df_g in [("outliers", df_noise), ("clustered", df_clustered)]:
        row = {
            "group": group,
            "n":     len(df_g),
            "pct":   round(len(df_g) / n_total * 100, 1),
        }

        # Quantitative
        for col in quanti_available:
            if col in df_g.columns:
                row[f"{col}_mean"]   = round(df_g[col].mean(), 1)
                row[f"{col}_median"] = round(df_g[col].median(), 1)

        # Triage
        if "triage" in df_g.columns:
            for level in sorted(df_work["triage"].dropna().unique()):
                row[f"triage_{int(level)}_pct"] = round(
                    (df_g["triage"] == level).mean() * 100, 1
                )

        # Measurement flags
        for k in available_measured:
            var = k.replace("is_", "").replace("_measured", "")
            row[f"{var}_measured_pct"] = round(df_g[k].mean() * 100, 1)

        # Status — % per modality
        for col in status_cols:
            for mod in sorted(df_g[col].dropna().unique()):
                row[f"{col}_{mod}_pct"] = round(
                    (df_g[col] == mod).mean() * 100, 1
                )

        rows.append(row)

    pd.DataFrame(rows).to_csv(
        os.path.join(out_dir, "outliers_external_summary.csv"), index=False
    )
    log.info(f"[{run_label}] Outlier external description complete.")

In [14]:
# ==============================================================================
# CALL — DESCRIPTION DES CLUSTERS - FULL DATASET
# ==============================================================================


# ── Paramètres ─────────────────────────────────────────────────────────────────
run_label = "s2_noweights"
mcs       = 1900   # ← définir mcs EN PREMIER

out_dir   = os.path.join(OUTPUT_DIR, run_label, f"description_mcs{mcs}")

# ── Chargement du CSV de clustering ───────────────────────────────────────────
df_clust = pd.read_csv(
    os.path.join(OUTPUT_DIR, run_label, f"clustering_mcs{mcs}.csv"),
    low_memory=False
)

labels = df_clust["cluster"].values
idx    = df_clust.index

# ── Description des clusters ───────────────────────────────────────────────────
describe_clusters(
    df        = df_clust,
    labels    = labels,
    idx       = idx,
    run_label = run_label,
    out_dir   = out_dir,
)

# ── Description des outliers ───────────────────────────────────────────────────
describe_outliers_external(
    df_clust  = df_clust,
    labels    = labels,
    idx       = idx,
    run_label = run_label,
    out_dir   = os.path.join(out_dir, "outliers"),
)



INFO | [s2_noweights] Description of 5 clusters (29769 patients)
INFO | Saved: Results/Regular_clustering/Full_dataset/s2_noweights/description_mcs1900/cluster_sizes.png
/tmp/ipykernel_2225830/3401689177.py:116: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, patch_artist=True, labels=[f"C{c}" for c in clusters])
/tmp/ipykernel_2225830/3401689177.py:116: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, patch_artist=True, labels=[f"C{c}" for c in clusters])
INFO | Saved: Results/Regular_clustering/Full_dataset/s2_noweights/description_mcs1900/admission_quanti.png
INFO | Saved: Results/Regular_clustering/Full_dataset/s2_noweights/description_mcs1900/admission_age_group.png
INFO | Saved: Results/Regular_c

Triage 1 (critiques) → 14% en bruit
→ profils atypiques, peu nombreux (170 patients)
→ leurs patterns de consommation ne ressemblent
  à aucun cluster existant
→ confirme ce qu'on avait discuté

Triage 2 → beaucoup d'outliers en valeur absolue
→ mais seulement 8% du total des triage 2
→ c'est la population la plus nombreuse donc
  même 8% représente beaucoup de patients

Triage 4-5 → très peu d'outliers
→ profils très homogènes et prévisibles
→ patients peu graves = consommation standardisée




Argument 1 — biais de sélection
Exclure les triage 1 = exclure les patients
les plus graves de ton analyse
→ tes clusters ne représenteraient plus
  la population réelle des urgences
→ biais majeur difficile à défendre
Argument 2 — 14% en bruit c'est pas si élevé
86% des triage 1 sont quand même assignés
à un cluster existant
→ la majorité a un profil reconnaissable
→ seuls 14% sont vraiment atypiques
Argument 3 — c'est une information en soi
"Les patients triage 1 sont sur-représentés
 parmi les outliers (14% vs 2-8% pour les autres)"
→ c'est une trouvaille clinique importante
→ ça veut dire que les critiques ont des
   profils de consommation plus hétérogènes
→ ce n'est pas du bruit — c'est de la réalité

Ce que je ferais à la place :
1. Les garder dans le clustering principal ✅

2. Dans la description des clusters, noter
   dans quel(s) cluster(s) tombent les 86%
   de triage 1 qui sont assignés

3. Mentionner explicitement dans ta thèse :
   "Les patients triage 1 présentent une
    hétérogénéité plus importante de leurs
    profils de consommation, avec 14% classés
    en bruit par HDBSCAN, contre 2-8%
    pour les autres niveaux de triage"
C'est beaucoup plus fort comme conclusion que de les exclure !

In [ ]:
# ==============================================================================
# CALL - DESCRIPTION CLUSTERS — SPLIT DATASET
# ==============================================================================

SPLIT_OUTPUT_DIR = "Results/Regular_clustering/Split_dataset"

# ── Hospitalized ───────────────────────────────────────────────────────────────
run_label_h = "s2_hospitalized"
mcs_h       = 1000

df_clust_h  = pd.read_csv(
    os.path.join(SPLIT_OUTPUT_DIR, run_label_h, f"clustering_mcs{mcs_h}.csv"),
    low_memory=False
)
labels_h = df_clust_h["cluster"].values
idx_h    = df_clust_h.index

describe_clusters(
    df        = df_clust_h,
    labels    = labels_h,
    idx       = idx_h,
    run_label = f"{run_label_h}_mcs{mcs_h}",
    out_dir   = os.path.join(SPLIT_OUTPUT_DIR, run_label_h, f"description_mcs{mcs_h}"),
)

describe_outliers_external(
    df_clust  = df_clust_h,
    labels    = labels_h,
    idx       = idx_h,
    run_label = f"{run_label_h}_mcs{mcs_h}",
    out_dir   = os.path.join(SPLIT_OUTPUT_DIR, run_label_h, f"description_mcs{mcs_h}", "outliers"),
)

# ── Discharged ─────────────────────────────────────────────────────────────────
run_label_d = "s2_discharged"
mcs_d       = 1800

df_clust_d  = pd.read_csv(
    os.path.join(SPLIT_OUTPUT_DIR, run_label_d, f"clustering_mcs{mcs_d}.csv"),
    low_memory=False
)
labels_d = df_clust_d["cluster"].values
idx_d    = df_clust_d.index

describe_clusters(
    df        = df_clust_d,
    labels    = labels_d,
    idx       = idx_d,
    run_label = f"{run_label_d}_mcs{mcs_d}",
    out_dir   = os.path.join(SPLIT_OUTPUT_DIR, run_label_d, f"description_mcs{mcs_d}"),
)

describe_outliers_external(
    df_clust  = df_clust_d,
    labels    = labels_d,
    idx       = idx_d,
    run_label = f"{run_label_d}_mcs{mcs_d}",
    out_dir   = os.path.join(SPLIT_OUTPUT_DIR, run_label_d, f"description_mcs{mcs_d}", "outliers"),
)